# Correcting a segmentation in the Cellpose GUI (one annotation round)

Exports one segmentation of the store written by notebook 01 for the Cellpose GUI, opens it, and records the round.
Each image of each round gets its own folder, `$MACROSEG_ANNOTATIONS/round_<NN>/<name>/` (see `macroseg.annotation`):

- `<name>.tif`: three image channels, shown as red, green and blue (the GUI keeps only the first three);
- `<name>_seg.npy`: the masks, loaded by the GUI; Cmd + S saves the corrections into it;
- `start_masks.tif`: the masks as exported, to count the changes;
- `annotation.json`: round, role (`train` / `test`), where image and masks come from, and after section 3
  what was changed and how well the starting masks scored.

In the GUI: right-click and draw to add a cell, Cmd + click to delete one, Cmd + S to save.
Review the **whole** image: untouched cells count as correct.

In [ ]:
from pathlib import Path

from spatialdata import read_zarr

from macroseg import paths
from macroseg.annotation import export_for_gui, finish, open_gui
from macroseg.config import load_config

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load_config(ROOT / "configs" / "exp201_test_crop.yaml")

ROUND = 1                                   # annotation round
ROLE = "train"                              # "train" or "test" (a test image is never used for training)
SEGMENTATION = "membrane_dapi"              # starting masks: a labels element of the store
CHANNELS = ["F480", "NaK_ATPase", "DAPI"]   # red, green, blue in the GUI
OVERWRITE = False                           # True: export again and discard the corrections saved in the GUI

## 1. Export image and masks
If the folder already holds `<name>_seg.npy`, nothing is exported (it may hold your corrections) unless `OVERWRITE = True`.

In [ ]:
sdata = read_zarr(cfg.zarr)
folder = export_for_gui(
    sdata, cfg, SEGMENTATION, CHANNELS, paths.path("MACROSEG_ANNOTATIONS"), ROUND, role=ROLE, overwrite=OVERWRITE
)
folder

## 2. Open the Cellpose GUI
A separate process: the notebook stays free, and closing the GUI does not stop the kernel.

In [ ]:
open_gui(folder);

## 3. After annotating: record the changes
Save in the GUI first (Cmd + S). `complete=False` if the image was not reviewed entirely.
Can be run again after further corrections.

In [ ]:
record = finish(folder, complete=True)
record["counts"], record["start_score"]